# Capítulo 1 · La fuente sísmica: de los catálogos a la ficha

**Modelación Sísmica y Tsunami (IRD-505) · Ingeniería en Riesgos de Desastres · ESPAM MFL**
Docente: Cristian Paliz Acosta, MSc.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/modelacion-sismica-tsunami-espam/blob/main/cuadernos/Cap1_fuente_sismica.ipynb)

Este cuaderno acompaña la **guía del trabajo sumativo de la Unidad 1** (Capítulo 1 del proyecto integrador). Descarga los datos de las agencias, los compara y produce las figuras, tablas y la **ficha de la fuente** que pide la guía:

| Paso | Qué produce | Sección de la guía |
|---|---|---|
| 1–2 | Sismo principal y tensor de momento del USGS; solución de Global CMT | 1.4, 1.5, 1.6 |
| 3 | Tablas de localización, tamaño y mecanismo; elección del plano | 1.4, 1.5, 1.6 |
| 4 | Pelotas de playa de ambas agencias (Figura 1.3) | 1.6 |
| 5 | Réplicas del primer mes, perfil de profundidad y L, W por réplicas (Figura 1.2) | 1.4, 1.7 |
| 6 | Mapa tectónico con fosa, placas y localidades (Figura 1.1) y velocidad de convergencia | 1.3 |
| 7 | L y W por tres vías, bordes de la falla y esquema de geometría (Figura 1.4) | 1.7 |
| 8 | D, U1 y U2, comparación con la falla finita | 1.8 |
| 9 | Ficha de la fuente con auditoría y CSV | 1.9 |

**Cómo se usa**

1. *Archivo → Guardar una copia en Drive* (así sus cambios quedan en su cuenta).
2. Ejecuten las celdas en orden (`Shift + Enter`). **Solo se editan las celdas marcadas con ✏️.**
3. Ejecuten el cuaderno **dos veces**: una para **Pedernales 2016** (caso común) y otra para **su evento asignado**. Al final, la celda de unión junta ambas fichas en un solo CSV.

**De dónde salen los datos.** Todo se descarga en el momento desde fuentes públicas y gratuitas: el servicio FDSN del USGS y el catálogo NDK de Global CMT. Si una agencia no responde, el cuaderno usa la copia de respaldo del repositorio del curso y lo avisa. Los valores que el cuaderno no puede descargar (modelo de falla finita, observaciones) se escriben a mano en las celdas ✏️, **siempre con su fuente**.

## Preparación del entorno
Instala ObsPy (solo la primera vez, menos de un minuto) y define utilidades. No hace falta editar.

In [ ]:
import importlib, subprocess, sys
try:
    import obspy
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "obspy"], check=True)
    import obspy

import io, os, json, math, gzip, datetime as dt, warnings
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from obspy import read_events
from obspy.imaging.beachball import beach
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_colwidth", 80)

REPO_RAW = "https://raw.githubusercontent.com/crispaliz/modelacion-sismica-tsunami-espam/main"
DATOS_LOCAL = os.environ.get("IRD505_DATOS", "")   # solo para pruebas locales del docente
FDSN = "https://earthquake.usgs.gov/fdsnws/event/1/query"
HOY = dt.date.today().isoformat()
KM_POR_GRADO = 111.195

def descargar(url, binario=False, timeout=90):
    """Descarga una URL; devuelve None si falla (sin detener el cuaderno)."""
    try:
        r = requests.get(url, timeout=timeout, headers={"User-Agent": "IRD505-ESPAM-docencia"})
        r.raise_for_status()
        return r.content if binario else r.text
    except Exception as err:
        print(f"   · No se pudo descargar {url[:90]}… ({type(err).__name__})")
        return None

def respaldo(ruta_relativa, binario=False):
    """Lee un archivo de la carpeta datos/ del repositorio del curso (o de una copia local)."""
    if DATOS_LOCAL:
        p = os.path.join(DATOS_LOCAL, ruta_relativa)
        if os.path.exists(p):
            return open(p, "rb").read() if binario else open(p, encoding="utf-8").read()
        return None
    return descargar(f"{REPO_RAW}/datos/{ruta_relativa}", binario=binario)

def a_km(lat, lon, lat0, lon0):
    e = (np.asarray(lon) - lon0) * KM_POR_GRADO * np.cos(np.radians(lat0))
    n = (np.asarray(lat) - lat0) * KM_POR_GRADO
    return e, n

def mw_desde_m0(m0):
    return 2 / 3 * (math.log10(m0) - 9.1)

def tipo_por_rake(rake):
    r = ((rake + 180) % 360) - 180
    if 45 <= r <= 135:   return "inversa"
    if -135 <= r <= -45: return "normal"
    return "de rumbo u oblicua"

# Eventos de la guía. Fecha UTC y coordenadas aproximadas: solo sirven para BUSCAR el sismo en los catálogos.
EVENTOS = {
    "Pedernales 2016": dict(fecha="2016-04-16", lat=0.38,   lon=-79.92,  Mw=7.8, tipo="Inversa de interplaca"),
    "Maule 2010":      dict(fecha="2010-02-27", lat=-36.12, lon=-72.90,  Mw=8.8, tipo="Inversa de interplaca"),
    "Illapel 2015":    dict(fecha="2015-09-16", lat=-31.57, lon=-71.67,  Mw=8.3, tipo="Inversa de interplaca"),
    "Iquique 2014":    dict(fecha="2014-04-01", lat=-19.61, lon=-70.77,  Mw=8.2, tipo="Inversa de interplaca"),
    "Pisco 2007":      dict(fecha="2007-08-15", lat=-13.39, lon=-76.60,  Mw=8.0, tipo="Inversa de interplaca"),
    "Camaná 2001":     dict(fecha="2001-06-23", lat=-16.26, lon=-73.64,  Mw=8.4, tipo="Inversa de interplaca"),
    "Tohoku 2011":     dict(fecha="2011-03-11", lat=38.30,  lon=142.37,  Mw=9.1, tipo="Inversa de interplaca"),
    "Nicaragua 1992":  dict(fecha="1992-09-02", lat=11.74,  lon=-87.34,  Mw=7.7, tipo="Sismo tsunami"),
    "Mentawai 2010":   dict(fecha="2010-10-25", lat=-3.49,  lon=100.08,  Mw=7.8, tipo="Sismo tsunami"),
    "Samoa 2009":      dict(fecha="2009-09-29", lat=-15.49, lon=-172.10, Mw=8.1, tipo="Normal, fuera de la fosa"),
    "Océano Índico 2012": dict(fecha="2012-04-11", lat=2.33, lon=93.06,  Mw=8.6, tipo="Rumbo"),
}
def slug(nombre):
    t = nombre.lower()
    for a, b in zip("áéíóúñ ", "aeioun_"):
        t = t.replace(a, b)
    return t
print("Entorno listo. ObsPy", obspy.__version__)

## Datos del grupo ✏️

- `EVENTO`: escriban exactamente uno de los nombres de la lista (primero **Pedernales 2016**, luego su evento asignado).
- `LOCALIDAD`: para Pedernales 2016, la ciudad de Pedernales. Para su evento asignado, **la localidad costera frente a la ruptura** que eligieron (será la localidad del grupo en los capítulos 2 a 4). Coordenadas en grados decimales, sur y oeste negativos.

In [ ]:
# ✏️ Datos del grupo
GRUPO     = "Grupo X"                       # p. ej. "Barre – Mora"
EVENTO    = "Pedernales 2016"               # uno de los nombres de la lista EVENTOS
LOCALIDAD = dict(nombre="Pedernales", lat=0.071, lon=-80.053)

EV = EVENTOS[EVENTO]
SLUG = slug(EVENTO)
print(f"{GRUPO} · {EVENTO} ({EV['tipo']}) · localidad: {LOCALIDAD['nombre']}")

## Paso 1 · Sismo principal y tensor de momento del USGS

Se consulta el servicio FDSN del USGS por fecha, zona y magnitud, y luego la página del evento: localización del **hipocentro**, Mw y el **tensor de momento W-phase** (centroide, M0 y los dos planos nodales). Si el evento tiene **modelo de falla finita**, se imprime el enlace: ábranlo y anoten los valores que pide el Paso 7.

In [ ]:
def usgs_detalle(ev):
    t0 = (pd.Timestamp(ev["fecha"]) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    t1 = (pd.Timestamp(ev["fecha"]) + pd.Timedelta(days=2)).strftime("%Y-%m-%d")
    url = (f"{FDSN}?format=geojson&starttime={t0}&endtime={t1}&latitude={ev['lat']}&longitude={ev['lon']}"
           f"&maxradius=3&minmagnitude={ev['Mw'] - 0.7:.1f}&orderby=magnitude&limit=1")
    txt = descargar(url)
    if txt:
        feats = json.loads(txt).get("features", [])
        if feats:
            det = descargar(f"{FDSN}?eventid={feats[0]['id']}&format=geojson")
            if det:
                return json.loads(det), "USGS (en línea)"
    txt = respaldo(f"respaldo/{SLUG}/usgs_detalle.json")
    if txt:
        print("   → Se usa la copia de respaldo del repositorio del curso.")
        return json.loads(txt), "USGS (copia de respaldo del curso)"
    return None, None

DET, ORIGEN_USGS = usgs_detalle(EV)
if DET is None:
    raise RuntimeError("No hay datos del USGS ni copia de respaldo. Revisen la conexión o avisen al docente.")
with open(f"usgs_{SLUG}.json", "w") as f:
    json.dump(DET, f)

P = DET["properties"]
lon_h, lat_h, z_h = DET["geometry"]["coordinates"]
USGS_ID = DET["id"]
HIPO = dict(fuente="USGS · hipocentro", tiempo_utc=pd.to_datetime(P["time"], unit="ms").strftime("%Y-%m-%d %H:%M:%S"),
            lat=lat_h, lon=lon_h, prof_km=z_h, Mw=P["mag"], tipo_mag=P.get("magType"))

def mejor_producto(nombre):
    prods = P.get("products", {}).get(nombre, [])
    return max(prods, key=lambda p: p.get("preferredWeight", 0)) if prods else None

def num(d, *claves):
    for k in claves:
        if d.get(k) not in (None, ""):
            return float(d[k])
    return None

mt = mejor_producto("moment-tensor")
USGS_MT = None
if mt:
    q = mt["properties"]
    USGS_MT = dict(
        fuente=f"USGS · tensor {q.get('derived-magnitude-type', q.get('beachball-type', ''))} ({mt.get('source', '')})",
        lat=num(q, "derived-latitude"), lon=num(q, "derived-longitude"), prof_km=num(q, "derived-depth"),
        M0=num(q, "scalar-moment"), Mw=num(q, "derived-magnitude"),
        NP1=(num(q, "nodal-plane-1-strike"), num(q, "nodal-plane-1-dip"), num(q, "nodal-plane-1-rake", "nodal-plane-1-slip")),
        NP2=(num(q, "nodal-plane-2-strike"), num(q, "nodal-plane-2-dip"), num(q, "nodal-plane-2-rake", "nodal-plane-2-slip")))

PAGINA = f"https://earthquake.usgs.gov/earthquakes/eventpage/{USGS_ID}"
print(f"Fuente: {ORIGEN_USGS} · consulta {HOY}")
print(f"Evento USGS {USGS_ID}: {P.get('title')}")
print(f"Página del evento: {PAGINA}")
print("Tensor de momento:", "sí" if USGS_MT else "no disponible")
print("Modelo de falla finita:", f"sí → {PAGINA}/finite-fault" if "finite-fault" in P.get("products", {}) else "no publicado por el USGS (busquen en SRCMOD o en artículos)")

## Paso 2 · Solución de Global CMT

Global CMT publica su catálogo completo como archivo de texto en formato NDK. El cuaderno lo descarga (unos 20 MB comprimidos; puede tardar uno o dos minutos), busca el sismo por hora y lo lee con ObsPy. **Ojo con las unidades:** el NDK trae M0 en dyn·cm; ObsPy ya lo entrega en **N·m** (dividido entre 10⁷).

In [ ]:
def gcmt_bloque(t_ref, mag_min):
    """Devuelve el bloque NDK (5 líneas) del sismo más cercano en tiempo a t_ref."""
    base = "www.ldeo.columbia.edu/~gcmt/projects/CMT/catalog"
    candidatos = [f"{pr}://{base}/jan76_dec{a:02d}.ndk.gz" for a in range(dt.date.today().year % 100, 19, -1) for pr in ("https", "http")]
    datos = None
    if os.path.exists("gcmt_catalogo.ndk.gz"):
        datos = open("gcmt_catalogo.ndk.gz", "rb").read()
    else:
        for url in candidatos:
            datos = descargar(url, binario=True, timeout=180)
            if datos:
                open("gcmt_catalogo.ndk.gz", "wb").write(datos)
                print("   Catálogo descargado:", url)
                break
    if datos:
        lineas = gzip.decompress(datos).decode("latin-1").splitlines()
        mejor, dmin = None, 1e9
        for i in range(0, len(lineas) - 4, 5):
            l1 = lineas[i]
            try:
                t = pd.Timestamp(l1[5:15].replace("/", "-") + " " + l1[16:26].strip())
            except Exception:
                continue
            d = abs((t - t_ref).total_seconds())
            if d < dmin:
                dmin, mejor = d, "\n".join(lineas[i:i + 5]) + "\n"
        if mejor and dmin < 180:
            return mejor, "Global CMT (en línea)"
    txt = respaldo(f"respaldo/{SLUG}/gcmt.ndk")
    if txt:
        print("   → Se usa la copia de respaldo del repositorio del curso.")
        return txt, "Global CMT (copia de respaldo del curso)"
    return None, None

BLOQUE, ORIGEN_GCMT = gcmt_bloque(pd.Timestamp(HIPO["tiempo_utc"]), EV["Mw"] - 0.7)
GCMT = None
if BLOQUE:
    open(f"gcmt_{SLUG}.ndk", "w").write(BLOQUE)
    e = read_events(f"gcmt_{SLUG}.ndk", format="NDK")[0]
    cen = [o for o in e.origins if o.origin_type == "centroid"][0]
    hip = [o for o in e.origins if o.origin_type == "hypocenter"][0]
    fm = e.preferred_focal_mechanism()
    npl = fm.nodal_planes
    mwc = [m.mag for m in e.magnitudes if m.magnitude_type == "Mwc"]
    GCMT = dict(fuente="Global CMT · centroide", tiempo_utc=str(cen.time)[:19].replace("T", " "),
                lat=cen.latitude, lon=cen.longitude, prof_km=cen.depth / 1000,
                M0=fm.moment_tensor.scalar_moment, Mw=mwc[0] if mwc else None,
                NP1=(npl.nodal_plane_1.strike, npl.nodal_plane_1.dip, npl.nodal_plane_1.rake),
                NP2=(npl.nodal_plane_2.strike, npl.nodal_plane_2.dip, npl.nodal_plane_2.rake),
                hipo_ref=(hip.latitude, hip.longitude, hip.depth / 1000), nombre=BLOQUE.splitlines()[1][:16].strip())
    print(f"Fuente: {ORIGEN_GCMT} · consulta {HOY} · evento {GCMT['nombre']}")
else:
    print("No se obtuvo la solución de Global CMT. Búsquenla en https://www.globalcmt.org/CMTsearch.html")
    print("y escriban los valores en la celda ✏️ siguiente (GCMT_MANUAL).")

In [ ]:
# ✏️ Solo si el paso anterior NO obtuvo Global CMT: copien los valores del sitio web (M0 en dyn·cm tal como lo da el sitio).
GCMT_MANUAL = None
# GCMT_MANUAL = dict(lat=0.0, lon=0.0, prof_km=0.0, M0_dyncm=0.0, Mw=0.0, NP1=(0, 0, 0), NP2=(0, 0, 0))

if GCMT is None and GCMT_MANUAL:
    GCMT = dict(GCMT_MANUAL, fuente="Global CMT · centroide (ingreso manual)", tiempo_utc="",
                M0=GCMT_MANUAL["M0_dyncm"] / 1e7)
    ORIGEN_GCMT = "Global CMT (sitio web, ingreso manual)"
print("Global CMT:", "listo" if GCMT else "pendiente")

## Paso 3 · Comparación entre agencias

Tres tablas para las secciones 1.4 a 1.6 del capítulo. Las diferencias entre agencias **se reportan y se explican**, no se esconden.

In [ ]:
fuentes = [x for x in (HIPO, USGS_MT, GCMT) if x]
loc = pd.DataFrame([{"Fuente": x["fuente"], "Hora UTC": x.get("tiempo_utc", ""), "Latitud (°)": round(x["lat"], 3),
                     "Longitud (°)": round(x["lon"], 3), "Profundidad (km)": round(x["prof_km"], 1),
                     "Tipo": "hipocentro" if x is HIPO else "centroide"} for x in fuentes])
print("Tabla 1.x · Localización (sección 1.4)"); display(loc)

tam = []
for x in (USGS_MT, GCMT):
    if x and x.get("M0"):
        tam.append({"Fuente": x["fuente"], "M0 (N·m)": f"{x['M0']:.3e}", "Mw reportada": x.get("Mw"),
                    "Mw recalculada": round(mw_desde_m0(x["M0"]), 2)})
tam = pd.DataFrame(tam)
print("\nTabla 1.x · Tamaño (sección 1.5) · Mw = (2/3)(log10 M0 − 9,1), M0 en N·m"); display(tam)
if USGS_MT and GCMT and USGS_MT.get("M0") and GCMT.get("M0"):
    dif = abs(USGS_MT["M0"] - GCMT["M0"]) / min(USGS_MT["M0"], GCMT["M0"]) * 100
    print(f"Diferencia de M0 entre agencias: {dif:.0f} %  → diferencia en Mw: {abs(mw_desde_m0(USGS_MT['M0']) - mw_desde_m0(GCMT['M0'])):.2f}")

mec = []
for x in (USGS_MT, GCMT):
    if x:
        for k in ("NP1", "NP2"):
            s_, d_, r_ = x[k]
            mec.append({"Fuente": x["fuente"], "Plano": k, "Rumbo φ (°)": s_, "Buzamiento δ (°)": d_, "Rake λ (°)": r_,
                        "Tipo por rake": tipo_por_rake(r_)})
mec = pd.DataFrame(mec)
print("\nTabla 1.x · Mecanismo focal (sección 1.6)"); display(mec)
sug = mec.loc[mec["Buzamiento δ (°)"].idxmin()]
print(f"Plano de MENOR buzamiento: {sug['Fuente']} {sug['Plano']} ({sug['Rumbo φ (°)']:.0f}/{sug['Buzamiento δ (°)']:.0f}/{sug['Rake λ (°)']:.0f}).")
print("En subducción interplaca suele ser el plano de falla. En sismos normales fuera de la fosa o de rumbo, la regla NO aplica: justifiquen con la geología del evento.")

### Elecciones del grupo ✏️

Estas decisiones son suyas y deben justificarse en el capítulo. Los valores por defecto son solo una sugerencia del cuaderno.

In [ ]:
# ✏️ Elecciones (escriban "USGS" o "GCMT", y "NP1" o "NP2")
AGENCIA_CENTROIDE = "GCMT"   # de qué agencia sale el centroide de la ficha (lat, lon, profundidad)
AGENCIA_M0        = "GCMT"   # de qué agencia sale el M0 adoptado
AGENCIA_PLANO     = "GCMT"   # de qué agencia sale el plano de falla
PLANO             = "NP1"    # cuál de los dos planos es la falla

SOL = {"USGS": USGS_MT, "GCMT": GCMT}
for nombre, ag in (("centroide", AGENCIA_CENTROIDE), ("M0", AGENCIA_M0), ("plano", AGENCIA_PLANO)):
    if SOL.get(ag) is None:
        raise ValueError(f"No hay solución de {ag} para el {nombre}; elijan la otra agencia.")
CEN = SOL[AGENCIA_CENTROIDE]
M0_ADOPT = SOL[AGENCIA_M0]["M0"]
RUMBO, BUZ, RAKE = SOL[AGENCIA_PLANO][PLANO]
otro = "NP2" if PLANO == "NP1" else "NP1"
print(f"Plano de falla: {AGENCIA_PLANO} {PLANO} → φ = {RUMBO:.0f}°, δ = {BUZ:.0f}°, λ = {RAKE:.0f}° (falla {tipo_por_rake(RAKE)})")
print(f"Plano auxiliar: {AGENCIA_PLANO} {otro} → {SOL[AGENCIA_PLANO][otro]}")
print(f"Centroide ({AGENCIA_CENTROIDE}): {CEN['lat']:.3f}°, {CEN['lon']:.3f}°, {CEN['prof_km']:.1f} km · M0 ({AGENCIA_M0}) = {M0_ADOPT:.3e} N·m (Mw {mw_desde_m0(M0_ADOPT):.2f})")
if abs(((RAKE + 180) % 360) - 180 - 90) > 20 and tipo_por_rake(RAKE) == "inversa":
    print("⚠️ El rake se aleja más de 20° de 90°: discutan la componente de rumbo y su efecto en U2.")

## Paso 4 · Pelotas de playa (Figura 1.3)

In [ ]:
sols = [(n, x) for n, x in (("USGS", USGS_MT), ("Global CMT", GCMT)) if x]
fig, axs = plt.subplots(1, len(sols), figsize=(4.6 * len(sols), 5.0))
axs = np.atleast_1d(axs)
for ax, (n, x) in zip(axs, sols):
    ax.add_collection(beach(list(x["NP1"]), xy=(0, 0), width=1.8, linewidth=1.2, facecolor="#C0392B"))
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    eleg = (n.startswith("Global") and AGENCIA_PLANO == "GCMT") or (n == "USGS" and AGENCIA_PLANO == "USGS")
    t = f"{n}\nNP1: {x['NP1'][0]:.0f}/{x['NP1'][1]:.0f}/{x['NP1'][2]:.0f}   NP2: {x['NP2'][0]:.0f}/{x['NP2'][1]:.0f}/{x['NP2'][2]:.0f}"
    if eleg:
        t += f"\nPlano de falla elegido: {PLANO}"
    ax.set_title(t, fontsize=10)
fig.suptitle(f"Figura 1.3 · Mecanismo focal · {EVENTO} (φ/δ/λ en grados)", fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.88]); plt.savefig(f"fig1_3_mecanismo_{SLUG}.png", dpi=200); plt.show()

## Paso 5 · Réplicas del primer mes (Figura 1.2) y dimensiones por réplicas

Se descargan los sismos del primer mes con magnitud mínima `MAG_MIN` alrededor del centroide. Las coordenadas se rotan al sistema de la falla (a lo largo del rumbo y en la dirección del buzamiento) y se mide la extensión de la nube entre los percentiles 5 y 95.

Cuidado al interpretar: la nube de réplicas suele ser **más grande** que el área que deslizó, y muchas profundidades de catálogo están fijadas (10 o 35 km). Por eso es solo una de las tres vías.

In [ ]:
# ✏️ Magnitud mínima de las réplicas y radio de búsqueda (km)
MAG_MIN = 4.0
L_STR = 10 ** (-2.477 + 0.585 * mw_desde_m0(M0_ADOPT))
W_STR = 10 ** (-0.882 + 0.351 * mw_desde_m0(M0_ADOPT))
RADIO_KM = max(150, 1.2 * L_STR)

t0 = pd.Timestamp(HIPO["tiempo_utc"]); t1 = t0 + pd.Timedelta(days=30)
url = (f"{FDSN}?format=csv&starttime={t0:%Y-%m-%dT%H:%M:%S}&endtime={t1:%Y-%m-%dT%H:%M:%S}"
       f"&latitude={CEN['lat']}&longitude={CEN['lon']}&maxradiuskm={RADIO_KM:.0f}&minmagnitude={MAG_MIN}&orderby=time-asc")
txt = descargar(url); ORIGEN_REP = "USGS (en línea)"
if not txt:
    txt = respaldo(f"respaldo/{SLUG}/replicas.csv"); ORIGEN_REP = "USGS (copia de respaldo del curso)"
REP = pd.read_csv(io.StringIO(txt)) if txt else pd.DataFrame()
if len(REP):
    REP = REP[(REP["mag"] >= MAG_MIN) & (pd.to_datetime(REP["time"]).dt.tz_localize(None) > t0 + pd.Timedelta(seconds=30))]
    REP.to_csv(f"replicas_{SLUG}.csv", index=False)
print(f"Réplicas M ≥ {MAG_MIN} en {RADIO_KM:.0f} km y 30 días: {len(REP)} · fuente: {ORIGEN_REP}")

us = np.array([math.sin(math.radians(RUMBO)), math.cos(math.radians(RUMBO))])        # a lo largo del rumbo
ub = np.array([math.sin(math.radians(RUMBO + 90)), math.cos(math.radians(RUMBO + 90))])  # dirección del buzamiento
L_REP = W_REP = None
if len(REP) >= 10:
    e, n = a_km(REP["latitude"].values, REP["longitude"].values, CEN["lat"], CEN["lon"])
    x_al, y_bz = e * us[0] + n * us[1], e * ub[0] + n * ub[1]
    p5x, p95x = np.percentile(x_al, [5, 95]); p5y, p95y = np.percentile(y_bz, [5, 95])
    L_REP = p95x - p5x
    W_REP = (p95y - p5y) / max(math.cos(math.radians(BUZ)), 0.2)
    print(f"Por réplicas: L ≈ {L_REP:.0f} km; W ≈ {W_REP:.0f} km (ancho horizontal {p95y - p5y:.0f} km / cos δ)")

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.8), gridspec_kw={"width_ratios": [1, 1.25]})
    sc = a1.scatter(REP["longitude"], REP["latitude"], c=REP["depth"], s=4 + 3 * (REP["mag"] - MAG_MIN) ** 2,
                    cmap="viridis_r", edgecolor="k", linewidth=0.2)
    a1.plot(HIPO["lon"], HIPO["lat"], "*", ms=18, color="gold", mec="k", label="Epicentro USGS")
    a1.plot(CEN["lon"], CEN["lat"], "s", ms=9, color="red", mec="k", label=f"Centroide {AGENCIA_CENTROIDE}")
    a1.plot(LOCALIDAD["lon"], LOCALIDAD["lat"], "^", ms=10, color="w", mec="k", label=LOCALIDAD["nombre"])
    a1.set_aspect(1 / math.cos(math.radians(CEN["lat"]))); a1.legend(fontsize=8, loc="best")
    a1.set_xlabel("Longitud (°)"); a1.set_ylabel("Latitud (°)"); a1.set_title("Réplicas del primer mes")
    plt.colorbar(sc, ax=a1, label="Profundidad (km)", shrink=0.8)
    a2.scatter(y_bz, REP["depth"], s=6, c="0.3")
    xx = np.array([-150, 150]); a2.plot(xx, CEN["prof_km"] + xx * math.tan(math.radians(BUZ)), "r--",
                                        label=f"plano δ = {BUZ:.0f}° por el centroide")
    a2.plot(0, CEN["prof_km"], "s", color="red", mec="k")
    a2.invert_yaxis(); a2.set_ylim(80, -5); a2.legend(fontsize=8)
    a2.set_xlabel("Distancia perpendicular al rumbo (km)   ← fosa · continente →"); a2.set_ylabel("Profundidad (km)")
    a2.set_title("Perfil de profundidad perpendicular a la fosa")
    fig.suptitle(f"Figura 1.2 · {EVENTO} · fuente: catálogo USGS ({ORIGEN_REP}, consulta {HOY})", fontsize=10)
    plt.tight_layout(); plt.savefig(f"fig1_2_replicas_{SLUG}.png", dpi=200); plt.show()
else:
    print("Muy pocas réplicas para estimar L y W por esta vía; bajen MAG_MIN o expliquen en el capítulo.")

## Paso 6 · Mapa tectónico (Figura 1.1) y velocidad de convergencia

Límites de placas del modelo PB2002 (Bird, 2003): en rojo los tramos de **subducción**. La velocidad de convergencia se toma del tramo de subducción más cercano al centroide. Para el mapa de Pedernales se marca solo Pedernales; para su evento, Pedernales queda fuera del mapa y se marca su localidad.

In [ ]:
def cargar_json(ruta_repo, url_alt):
    t = respaldo(ruta_repo) or descargar(url_alt)
    return json.loads(t) if t else None

LIM = cargar_json("pb2002_limites.geojson", "https://raw.githubusercontent.com/fraxen/tectonicplates/master/GeoJSON/PB2002_boundaries.json")
txt = respaldo("pb2002_pasos.csv")
PASOS = pd.read_csv(io.StringIO(txt)) if txt else pd.DataFrame()
COSTA = cargar_json("ne_50m_coastline.geojson", "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_50m_coastline.geojson")

def ajustar_lon(lon, lon0):
    return (np.asarray(lon) - lon0 + 180) % 360 - 180 + lon0

lat0, lon0 = CEN["lat"], CEN["lon"]
m = max(6.0, L_STR / 111 + 4)
ext = (lon0 - m * 1.3, lon0 + m * 1.3, lat0 - m, lat0 + m)
fig, ax = plt.subplots(figsize=(9, 8.5))
if COSTA:
    for f in COSTA["features"]:
        g = f["geometry"]; ls = [g["coordinates"]] if g["type"] == "LineString" else g["coordinates"]
        for l in ls:
            a = np.array(l); lo = ajustar_lon(a[:, 0], lon0)
            if ((lo > ext[0] - 5) & (lo < ext[1] + 5) & (a[:, 1] > ext[2] - 5) & (a[:, 1] < ext[3] + 5)).any():
                lo = np.where(np.abs(np.diff(lo, prepend=lo[0])) > 180, np.nan, lo)
                ax.plot(lo, a[:, 1], color="0.25", lw=0.8)
if LIM:
    for f in LIM["features"]:
        a = np.array(f["geometry"]["coordinates"]); lo = ajustar_lon(a[:, 0], lon0)
        lo = np.where(np.abs(np.diff(lo, prepend=lo[0])) > 180, np.nan, lo)
        ax.plot(lo, a[:, 1], color="0.55", lw=1, ls="--")
VEL = None
if len(PASOS):
    sub = PASOS[PASOS["clase"] == "SUB"].copy()
    l1, l2 = ajustar_lon(sub["lon1"], lon0), ajustar_lon(sub["lon2"], lon0)
    for x1, y1, x2, y2 in zip(l1, sub["lat1"], l2, sub["lat2"]):
        if abs(x2 - x1) < 10:
            ax.plot([x1, x2], [y1, y2], color="#C0392B", lw=2.4)
    dd = np.hypot((ajustar_lon(sub["lon"], lon0) - lon0) * math.cos(math.radians(lat0)), sub["lat"] - lat0) * KM_POR_GRADO
    i = dd.idxmin()
    if dd[i] < 400:
        VEL = dict(limite=sub.loc[i, "limite"], total=sub.loc[i, "vel_mm_a"], conv=-sub.loc[i, "vel_div_mm_a"],
                   az=sub.loc[i, "vel_az"], dist=dd[i])
ax.plot([], [], color="#C0392B", lw=2.4, label="Subducción (PB2002)")
ax.plot([], [], color="0.55", lw=1, ls="--", label="Otros límites de placa (PB2002)")
ax.plot(HIPO["lon"], HIPO["lat"], "*", ms=20, color="gold", mec="k", label=f"Epicentro {EVENTO}")
ax.plot(LOCALIDAD["lon"], LOCALIDAD["lat"], "^", ms=11, color="w", mec="k", label=LOCALIDAD["nombre"])
ax.annotate(LOCALIDAD["nombre"], (LOCALIDAD["lon"], LOCALIDAD["lat"]), xytext=(8, -12), textcoords="offset points", fontsize=9)
ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); ax.set_aspect(1 / math.cos(math.radians(lat0)))
# Escala gráfica y norte
km = 200 if m > 7 else 100
dlon = km / (KM_POR_GRADO * math.cos(math.radians(lat0)))
xb, yb = ext[0] + 0.06 * (ext[1] - ext[0]), ext[2] + 0.06 * (ext[3] - ext[2])
ax.plot([xb, xb + dlon], [yb, yb], color="k", lw=4); ax.text(xb + dlon / 2, yb + 0.02 * (ext[3] - ext[2]), f"{km} km", ha="center", fontsize=9)
xn, yn = ext[1] - 0.07 * (ext[1] - ext[0]), ext[3] - 0.14 * (ext[3] - ext[2])
ax.annotate("N", xy=(xn, yn + 0.08 * (ext[3] - ext[2])), xytext=(xn, yn), ha="center", fontsize=12, fontweight="bold",
            arrowprops=dict(arrowstyle="-|>", color="k", lw=2))
ax.set_xlabel("Longitud (°)"); ax.set_ylabel("Latitud (°)"); ax.legend(fontsize=8, loc="lower right")
ax.set_title(f"Figura 1.1 · Marco tectónico · {EVENTO}\nLímites: Bird (2003), PB2002 · Costa: Natural Earth · Epicentro: USGS", fontsize=10)
plt.tight_layout(); plt.savefig(f"fig1_1_mapa_tectonico_{SLUG}.png", dpi=200); plt.show()
if VEL:
    print(f"Tramo de subducción más cercano ({VEL['dist']:.0f} km): límite {VEL['limite']}; velocidad relativa {VEL['total']:.0f} mm/a,"
          f" componente convergente {VEL['conv']:.0f} mm/a (Bird, 2003). Compárenla con la que reporta el USGS para el evento.")
else:
    print("No hay un tramo de subducción a menos de 400 km: es esperable en sismos de rumbo o intraplaca. Usen la fuente que corresponda.")

## Paso 7 ✏️ · Dimensiones por tres vías, bordes y geometría (Figura 1.4)

1. **Falla finita:** abran el enlace del Paso 1 (o SRCMOD / el artículo que usen) y anoten L y W del **área con deslizamiento significativo**, no de toda la malla, junto con el deslizamiento medio y máximo.
2. **Réplicas:** calculado en el Paso 5.
3. **Escala:** Strasser et al. (2010) para interplaca: log10 L = −2,477 + 0,585 Mw y log10 W = −0,882 + 0,351 Mw.

Luego elijan L y W y justifíquenlo. Bordes: `z_sup = z_c − (W/2)·sen δ` y `z_inf = z_c + (W/2)·sen δ`.

In [ ]:
# ✏️ Datos del modelo de falla finita (déjenlos en None si no existen y explíquenlo en el capítulo)
FF = dict(L_km=None, W_km=None, D_medio_m=None, D_max_m=None, fuente="")
# Ej.: FF = dict(L_km=140, W_km=80, D_medio_m=0.7, D_max_m=2.5, fuente="Autor (año) o USGS finite fault, consulta AAAA-MM-DD")

# ✏️ Dimensiones adoptadas y su justificación (por defecto: escala de Strasser)
L_ADOPT = round(L_STR)
W_ADOPT = round(W_STR)
JUSTIF_LW = "Escriban aquí por qué adoptan estos valores."
Z_SUP_FIJO = None      # solo si z_sup sale negativo: un valor entre 1 y 3 km, explicado en el capítulo

dims = pd.DataFrame([
    {"Vía": "Modelo de falla finita (área con deslizamiento significativo)", "L (km)": FF["L_km"], "W (km)": FF["W_km"], "Fuente": FF["fuente"] or "—"},
    {"Vía": "Réplicas del primer mes (P5–P95)", "L (km)": None if L_REP is None else round(L_REP), "W (km)": None if W_REP is None else round(W_REP), "Fuente": f"Catálogo USGS, consulta {HOY}"},
    {"Vía": "Escala de Strasser et al. (2010)", "L (km)": round(L_STR), "W (km)": round(W_STR), "Fuente": f"Mw = {mw_desde_m0(M0_ADOPT):.2f}"},
    {"Vía": "ADOPTADO", "L (km)": L_ADOPT, "W (km)": W_ADOPT, "Fuente": JUSTIF_LW},
])
print("Tabla 1.x · Dimensiones de la ruptura (sección 1.7)"); display(dims)

ZC = CEN["prof_km"]
Z_SUP = ZC - W_ADOPT / 2 * math.sin(math.radians(BUZ))
Z_INF = ZC + W_ADOPT / 2 * math.sin(math.radians(BUZ))
print(f"z_sup = {ZC:.1f} − {W_ADOPT / 2:.1f}·sen {BUZ:.0f}° = {Z_SUP:.2f} km;   z_inf = {Z_INF:.2f} km")
if Z_SUP < 0:
    if Z_SUP_FIJO is None:
        raise ValueError("z_sup es negativo: la falla saldría por encima del fondo. Fijen Z_SUP_FIJO entre 1 y 3 km y explíquenlo.")
    print(f"⚠️ z_sup negativo: se fija en {Z_SUP_FIJO} km (decisión del grupo, a justificar)."); Z_SUP = Z_SUP_FIJO
    Z_INF = Z_SUP + W_ADOPT * math.sin(math.radians(BUZ))

# Figura 1.4: geometría en corte perpendicular al rumbo
h = W_ADOPT * math.cos(math.radians(BUZ))
x_sup, x_inf = -h / 2, h / 2
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.axhline(0, color="#2E86C1", lw=2); ax.text(0.01, 0.97, "fondo marino / superficie (0 km)", color="#2E86C1", fontsize=9, transform=ax.transAxes, va="top")
ax.plot([x_sup, x_inf], [Z_SUP, Z_INF], color="k", lw=4)
ax.plot(0, (Z_SUP + Z_INF) / 2, "s", color="red", ms=9, mec="k")
ax.annotate(f"centroide\nz_c = {ZC:.1f} km", (0, (Z_SUP + Z_INF) / 2), xytext=(-120, -42), textcoords="offset points", fontsize=9)
ax.annotate(f"borde superior\nz_sup = {Z_SUP:.1f} km", (x_sup, Z_SUP), xytext=(-110, -5), textcoords="offset points", fontsize=9,
            arrowprops=dict(arrowstyle="->"))
ax.annotate(f"borde inferior\nz_inf = {Z_INF:.1f} km", (x_inf, Z_INF), xytext=(15, 0), textcoords="offset points", fontsize=9,
            arrowprops=dict(arrowstyle="->"))
ax.plot([x_sup, x_sup + h * 0.35], [Z_SUP, Z_SUP], color="0.5", ls=":")
ax.text(x_sup + h * 0.12, Z_SUP + 0.06 * Z_INF + 1, f"δ = {BUZ:.0f}°", fontsize=10)
ax.text((x_sup + x_inf) / 2 - h * 0.1, (Z_SUP + Z_INF) / 2 - 0.12 * Z_INF - 2, f"W = {W_ADOPT:.0f} km", fontsize=10, rotation=-BUZ * 0.6)
ax.invert_yaxis(); ax.set_ylim(Z_INF + 12, -6); ax.set_xlim(x_sup - h * 0.7, x_inf + h * 0.7)
ax.set_xlabel("Distancia perpendicular al rumbo (km)   ← fosa · continente →"); ax.set_ylabel("Profundidad (km)")
ax.set_title(f"Figura 1.4 · Geometría de la falla adoptada · {EVENTO} (L = {L_ADOPT:.0f} km a lo largo del rumbo)", fontsize=10)
ax.set_aspect("equal")
plt.tight_layout(); plt.savefig(f"fig1_4_geometria_{SLUG}.png", dpi=200); plt.show()

## Paso 8 ✏️ · Deslizamiento y sus componentes

`D = M0 / (μ·L·W)` con L y W en **metros** y μ en **pascales**; `U1 = D·cos λ` (rumbo) y `U2 = D·sen λ` (buzamiento, la que levanta o hunde el fondo). Adopten μ entre 30 y 40 GPa, salvo que el tipo de evento justifique otro valor (sismos tsunami: cuña de acreción, rigidez menor).

In [ ]:
# ✏️ Módulo de cizalla adoptado (GPa) y su justificación
MU_GPA = 30.0
JUSTIF_MU = "Valor estándar para la interfaz (30–40 GPa)."

D = M0_ADOPT / (MU_GPA * 1e9 * L_ADOPT * 1e3 * W_ADOPT * 1e3)
U1, U2 = D * math.cos(math.radians(RAKE)), D * math.sin(math.radians(RAKE))
print(f"D = {M0_ADOPT:.3e} / ({MU_GPA:.0f}×10⁹ · {L_ADOPT * 1e3:.0f} · {W_ADOPT * 1e3:.0f}) = {D:.2f} m")
print(f"U1 = D·cos {RAKE:.0f}° = {U1:.2f} m   ·   U2 = D·sen {RAKE:.0f}° = {U2:.2f} m   ·   D·sen δ = {D * math.sin(math.radians(BUZ)):.2f} m")
if FF["D_medio_m"]:
    print(f"Falla finita: D medio = {FF['D_medio_m']} m, D máximo = {FF['D_max_m']} m ({FF['fuente']})")
    if FF["D_max_m"] and D > FF["D_max_m"]:
        print("⚠️ El D uniforme SUPERA el máximo del modelo de falla finita: el área adoptada es probablemente menor que la que deslizó,"
              " o el μ / M0 difieren. Discútanlo en la sección 1.8.")
    D_area_ff = M0_ADOPT / (MU_GPA * 1e9 * FF["L_km"] * 1e3 * FF["W_km"] * 1e3) if FF["L_km"] and FF["W_km"] else None
    if D_area_ff:
        print(f"Con el área de la falla finita ({FF['L_km']} × {FF['W_km']} km) el mismo M0 da D = {D_area_ff:.2f} m.")
else:
    print("⚠️ Sin datos de falla finita: la guía pide comparar D con el deslizamiento medio y máximo publicados.")

## Paso 9 · Ficha de la fuente, auditoría y CSV

La ficha lleva **un solo valor por parámetro**, con unidad, fuente y método. La auditoría revisa los errores frecuentes de la guía. El CSV es el archivo que se carga en el cuaderno de Okada (`FICHA_CSV`).

In [ ]:
src_cen = f"{SOL[AGENCIA_CENTROIDE]['fuente']} (consulta {HOY})"
src_pl = f"{SOL[AGENCIA_PLANO]['fuente']} {PLANO} (consulta {HOY})"
FICHA = [
    ("lat", round(CEN["lat"], 4), "grados", src_cen, "Centroide adoptado"),
    ("lon", round(CEN["lon"], 4), "grados", src_cen, "Centroide adoptado"),
    ("z_sup", round(Z_SUP, 2), "km", "Cálculo propio", f"z_c − (W/2)·sen δ con z_c = {ZC:.1f} km" if Z_SUP_FIJO is None else "Fijado por el grupo (z_sup < 0)"),
    ("rumbo", round(RUMBO, 1), "grados", src_pl, "Plano de falla elegido"),
    ("buz", round(BUZ, 1), "grados", src_pl, "Plano de falla elegido"),
    ("rake", round(RAKE, 1), "grados", src_pl, "Plano de falla elegido"),
    ("L", float(L_ADOPT), "km", "Comparación de tres vías", JUSTIF_LW),
    ("W", float(W_ADOPT), "km", "Comparación de tres vías", JUSTIF_LW),
    ("D", round(D, 3), "m", "Cálculo propio", f"M0/(μLW) con M0 de {AGENCIA_M0}"),
    ("mu", MU_GPA, "GPa", "Valor adoptado", JUSTIF_MU),
    ("M0", float(f"{M0_ADOPT:.4e}"), "N·m", f"{SOL[AGENCIA_M0]['fuente']} (consulta {HOY})", "M0 adoptado"),
    ("Mw", round(mw_desde_m0(M0_ADOPT), 2), "—", "Cálculo propio", "(2/3)(log10 M0 − 9,1)"),
]
ficha = pd.DataFrame(FICHA, columns=["parametro", "valor", "unidad", "fuente", "metodo"])
ficha.insert(0, "evento", EVENTO)
print(f"Tabla 1.x · Ficha de la fuente · {EVENTO}"); display(ficha.astype({"valor": object}))

aud = []
def chk(n, ok, nota, aviso=False):
    aud.append({"Control": n, "Estado": "✅" if ok else ("⚠️" if aviso else "❌"), "Comentario": nota})
chk("Un valor numérico en cada parámetro", ficha["valor"].notna().all(), "Sin celdas vacías ni rangos")
chk("La coordenada es un centroide, no el epicentro",
    not (abs(CEN["lat"] - HIPO["lat"]) < 1e-3 and abs(CEN["lon"] - HIPO["lon"]) < 1e-3), "Okada usa el centroide de la falla")
chk("z_sup ≥ 0", Z_SUP >= 0, "Si es negativo, la falla sale por encima del fondo")
chk("Plano de falla de bajo buzamiento (si es interplaca)", BUZ <= 35 or "interplaca" not in EV["tipo"].lower(),
    "En interplaca el plano real suele tener δ ≤ 35°", aviso=True)
chk("L dentro de ×2 de la escala", 0.5 <= L_ADOPT / L_STR <= 2, f"Strasser: {L_STR:.0f} km", aviso=True)
chk("W dentro de ×2 de la escala", 0.5 <= W_ADOPT / W_STR <= 2, f"Strasser: {W_STR:.0f} km", aviso=True)
chk("D no supera el máximo de la falla finita", not (FF["D_max_m"] and D > FF["D_max_m"]), "Si lo supera, discútanlo", aviso=True)
chk("Datos de falla finita ingresados", bool(FF["D_medio_m"]), "La guía pide compararlos", aviso=True)
chk("Justificaciones escritas", "Escriban aquí" not in JUSTIF_LW, "Completen JUSTIF_LW", aviso=True)
display(pd.DataFrame(aud))

nombre = f"ficha_{SLUG}.csv"
ficha.to_csv(nombre, index=False)
print("Guardado:", nombre)

## Unión de fichas y descarga de archivos

Después de ejecutar el cuaderno para **Pedernales 2016 y para su evento**, esta celda une las fichas en `fichas_<grupo>.csv` (el CSV que se entrega) y descarga todos los archivos. Si reiniciaron Colab entre una corrida y otra, suban primero la ficha anterior con el ícono de carpeta de la izquierda.

In [ ]:
import glob
fichas = sorted(glob.glob("ficha_*.csv"))
if fichas:
    todo = pd.concat([pd.read_csv(f) for f in fichas], ignore_index=True)
    salida = f"fichas_{GRUPO.replace(' ', '_').replace('–', '-')}.csv"
    todo.to_csv(salida, index=False)
    print("Fichas unidas:", ", ".join(todo["evento"].unique()), "→", salida)
    if "Pedernales 2016" not in set(todo["evento"]):
        print("⚠️ Falta la ficha de Pedernales 2016 (caso común).")
archivos = fichas + glob.glob("fichas_*.csv") + glob.glob("fig1_*.png") + glob.glob("replicas_*.csv")
print("Archivos:", *sorted(set(archivos)), sep="\n  ")
try:
    from google.colab import files
    for a in sorted(set(archivos)):
        files.download(a)
except ImportError:
    pass

## Referencias

- Bird, P. (2003). An updated digital model of plate boundaries. *Geochemistry, Geophysics, Geosystems, 4*(3), 1027. https://doi.org/10.1029/2001GC000252
- Ekström, G., Nettles, M., y Dziewoński, A. M. (2012). The global CMT project 2004–2010: Centroid-moment tensors for 13,017 earthquakes. *Physics of the Earth and Planetary Interiors, 200–201*, 1–9. https://doi.org/10.1016/j.pepi.2012.04.002
- Hanks, T. C., y Kanamori, H. (1979). A moment magnitude scale. *Journal of Geophysical Research, 84*(B5), 2348–2350.
- Strasser, F. O., Arango, M. C., y Bommer, J. J. (2010). Scaling of the source dimensions of interface and intraslab subduction-zone earthquakes with moment magnitude. *Seismological Research Letters, 81*(6), 941–950.
- U.S. Geological Survey. Earthquake Catalog (servicio FDSN). https://earthquake.usgs.gov/fdsnws/event/1/
- Límites de placa en GeoJSON: Ahlenius, H. (fraxen/tectonicplates), ODC-By. Línea de costa: Natural Earth (dominio público).

En el capítulo, citen además la página del evento del USGS y el catálogo Global CMT **con la fecha de consulta** que imprimió el cuaderno.